# ESG Data Automation Validation
The objective of this project is to simulate a real-world scenario: a company has three production facilities (Milano, Lione, Roma) and each plant manager sends an Excel file with the year's energy consumption data. The goal is to merge these datasets, clean them, compute GHG emissions (Scope 1 & 2) and export a structured Excel report ready for sustainability disclosure.

The project is structured in four phases:
1. **Data Loading & Exploration** — import and first look at the raw files
2. **Data Cleaning** — handle missing values, duplicates, inconsistent formats
3. **Emissions Calculation** — apply emission factors to compute kgCO₂e (the ESG logic)
4. **Export & Reporting** — generate a formatted Excel output with summary and chart

## Phase 1 — Data Loading & Exploration

Before any cleaning or calculation, we load the three raw Excel files and run an initial exploratory analysis to understand the structure and quality of the data: shape, data types, missing values and duplicates.

We start by importing the necessary libraries and loading the three datasets. A Sweetviz report is generated for each file to get an automated visual overview of distributions, missing values and anomalies.

In [2]:
import pandas as pd
import numpy as np
import sweetviz as sv

from config import mappa_categorie, mappa_risorsa_categoria, mappa_risorsa_unita, valori_non_validi, mappa_unita, mappa_risorsa, lookup_categoria

df_roma    = pd.read_excel("data\\consumi_roma.xlsx")
df_lione  = pd.read_excel("data\\consumi_lione.xlsx")
df_milano  = pd.read_excel("data\\consumi_milano.xlsx")

'''#Looking at the data
report_roma = sv.analyze([df_roma, "Roma"])
report_roma.show_html("report_roma.html")

report_milano = sv.analyze([df_milano, "Milano"])
report_milano.show_html("report_milano.html")

report_lione = sv.analyze([df_lione, "Lione"])
report_lione.show_html("report_lione.html")'''

pd.set_option('display.expand_frame_repr', False)

We now inspect each dataset manually: shape (rows and columns), data types per column, percentage of missing values, and number of duplicates. This gives us a clear picture of what needs to be fixed before merging.

In [3]:
# Shape
print(df_roma.shape, '   |   ', df_milano.shape, '   |   ', df_lione.shape )

(134, 5)    |    (133, 5)    |    (137, 5)


In [4]:
# Data types
df_dtypes = pd.DataFrame({
    'Roma': df_roma.dtypes,
    'Milano': df_milano.dtypes,
    'Lione': df_lione.dtypes
})
df_dtypes

,Roma,Milano,Lione
Data,object,object,object
Categoria,object,object,object
Risorsa,object,object,object
Quantità,float64,float64,float64
Unità,object,object,object


In [5]:
# Check amount of null values
df_missing_pct = pd.DataFrame({
    'Roma int':   df_roma.isnull().sum(),
    'Roma %':   (df_roma.isnull().sum()   / len(df_roma)   * 100).round(1),
    'Milano int': df_milano.isnull().sum(),
    'Milano %': (df_milano.isnull().sum() / len(df_milano) * 100).round(1),
    'Lione int':  df_lione.isnull().sum(),
    'Lione %':  (df_lione.isnull().sum()  / len(df_lione)  * 100).round(1),
})
df_missing_pct

,Roma int,Roma %,Milano int,Milano %,Lione int,Lione %
Data,11,8.2,13,9.8,10,7.3
Categoria,3,2.2,3,2.3,3,2.2
Risorsa,4,3.0,4,3.0,4,2.9
Quantità,9,6.7,13,9.8,10,7.3
Unità,4,3.0,3,2.3,3,2.2


In [6]:
df_milano.head(10)

,Data,Categoria,Risorsa,Quantità,Unità
0,14/09/2026,Raffreddamento,Elettricità,382.17,kWh
1,30/07/2025,ENERGIA,Elettricità,2354.47,kilowattora
2,2025-09-05,RAFFREDDAMENTO,Gas refrigerante R410A,6.51,Kg
3,06/03/2024,Raffreddamento,Gas refrigerante R410A,2.79,KG
4,19/01/25,ACQUA,Acqua di rete,95.25,m3
5,25/03/2024,riscaldamento,Gasolio,220.13,litri
6,06/12/2026,Riscald.,Gasolio,272.91,L
7,NaN,NaN,NaN,NaN,NaN
8,2024-12-21,Raffreddamento,Gas refrigerante R410A,6.39,Kg
9,14/04/24,Fleet,Gas Naturale,NaN,mc


We check how many duplicate rows are present in each file. Duplicates are common when data is exported from ERP systems or compiled manually — they must be removed before any aggregation.

In [7]:
print(f"Roma:   {df_roma.duplicated().sum()} duplicati")
print(f"Milano: {df_milano.duplicated().sum()} duplicati")
print(f"Lione:  {df_lione.duplicated().sum()} duplicati")

Roma:   9 duplicati
Milano: 9 duplicati
Lione:  10 duplicati


Before cleaning the data, we verify that the column names are consistent across all three files — any difference in naming or formatting would cause misalignment in the final DataFrame.

In [8]:
# Modify the columns' names if they have some differences in format
lista_df = [df_milano, df_lione, df_roma]

for df in lista_df:
    df.columns = df.columns.str.replace(' ', '_').str.title().str.strip()

## Phase 2 — Data Cleaning

Raw facility data is rarely clean. In this phase we handle all the inconsistencies found in Phase 1:
- **Empty rows** — dropped immediately
- **Inconsistent text** — category names, resource names and units are standardized using lookup dictionaries (e.g. "KWH", "kwh", "kilowattora" → "kWh")
- **Missing values** — inferred where possible from other columns (e.g. missing Category inferred from Resource), logged as anomalies otherwise
- **Invalid quantities** — negative values, zeros and non-numeric strings are flagged and removed
- **Unit mismatches** — rows where the unit does not match the expected unit for that resource are flagged (e.g. Gas Naturale in Litri instead of m3)
- **Dates** — parsed to a consistent datetime format and sorted chronologically

In [9]:
lista_df = [df_roma, df_milano, df_lione]
nomi     = ['Roma',  'Milano',  'Lione']

for df, nome in zip(lista_df, nomi):
    
    # drops rows with all missing values
    df.dropna(how='all', inplace=True)

    # For every column categoria and unità fixes the format and fill if blanks
    df['Categoria'] = df['Categoria'].str.strip().map(mappa_categorie).fillna(df['Categoria'])
    df['Risorsa'] = df['Risorsa'].str.strip().map(mappa_risorsa).fillna(df['Risorsa'])
    df['Unità']     = df['Unità'].str.strip().map(mappa_unita).fillna(df['Unità'])
    df['Unità']     = df['Unità'].replace(valori_non_validi, np.nan)
    
    # definisco condizioni di sostituzione
    mask_cat = df['Categoria'].isna() & df['Risorsa'].notna()
    mask_unita = df['Unità'].isna() & df['Risorsa'].notna()
    
    # se le condizioni di sostituzione sono rispettate allora sostituisco facendo riferimento ai dizionari contenenti i gli abbinamenti
    df.loc[mask_cat, 'Categoria'] = df.loc[mask_cat, 'Risorsa'].map(lookup_categoria)
    df.loc[mask_unita, 'Unità'] = df.loc[mask_unita, 'Risorsa'].map(mappa_risorsa_unita)
    
    # nel caso ci siano valori non validi non come NaN, vengono sostituiti da NaN
    df['Quantità'] = df['Quantità'].replace(valori_non_validi, np.nan)

    # Drop delle righe con quantità negative o nulle, quando le unità di misura sono sbagliate, e quando ci sono missing values e duplicati
    df.drop(df[df['Quantità'] <= 0].index, inplace=True)
    df.drop(df[df['Unità'] != df['Risorsa'].map(mappa_risorsa_unita)].index, inplace=True)  # se le unità di misura della risorsa sono sbagliate droppo la riga
    df.dropna(subset=['Data', 'Quantità', 'Categoria'], inplace=True)
    df.drop_duplicates(inplace=True)


After cleaning, we verify that all categories, resources and units have been correctly standardized by inspecting the unique combinations present in each dataset.

In [10]:
for nome, df in zip(nomi, lista_df):
    print(f"\n[{nome}]")
    print(
        df.groupby(['Categoria', 'Risorsa', 'Unità'])
        .size()
        .reset_index(name='Conteggio')
        .to_string(index=False)
    )


[Roma]
       Categoria                Risorsa Unità  Conteggio
           Acqua          Acqua di rete    m3         18
         Energia            Elettricità   kWh         17
Flotta aziendale                Benzina Litri         11
Flotta aziendale           Gas Naturale    m3          5
Flotta aziendale                Gasolio Litri          3
  Raffreddamento            Elettricità   kWh          9
  Raffreddamento Gas refrigerante R410A    kg          6
         Rifiuti       Rifiuti speciali    kg         10
         Rifiuti         Rifiuti urbani    kg          8
   Riscaldamento               Biomassa    kg          3
   Riscaldamento           Gas Naturale    m3          8
   Riscaldamento                Gasolio Litri          6

[Milano]
       Categoria                Risorsa Unità  Conteggio
           Acqua          Acqua di rete    m3         12
         Energia            Elettricità   kWh         17
Flotta aziendale                Benzina Litri          2
Flotta aziend

Finally, dates are parsed from their various raw formats (dd/mm/yyyy, yyyy-mm-dd, etc.) into a consistent datetime and rows are sorted chronologically.

All three files are now clean and consistent. We add an `Impianto` column to each one to preserve the source facility, then concatenate them into a single unified DataFrame.

In [14]:
for nome, df in zip(nomi, lista_df):
    df['Data'] = pd.to_datetime(
        df['Data'], dayfirst=True, format='mixed').dt.date
    df.sort_values(by='Data', inplace = True)
    df.reset_index(drop=True, inplace=True)
    df['Impianto'] = nome

df = pd.concat([df_lione, df_milano, df_roma])
df

,Data,Categoria,Risorsa,Quantità,Unità,Impianto
0,2024-01-03,Flotta aziendale,Gas Naturale,20.80,m3,Lione
1,2024-01-09,Acqua,Acqua di rete,103.28,m3,Lione
2,2024-01-15,Energia,Elettricità,2992.19,kWh,Lione
3,2024-01-23,Flotta aziendale,Gas Naturale,30.03,m3,Lione
4,2024-02-16,Riscaldamento,Gas Naturale,725.08,m3,Lione
...,...,...,...,...,...,...
99,2026-12-03,Raffreddamento,Gas refrigerante R410A,7.47,kg,Roma
100,2026-12-14,Flotta aziendale,Benzina,53.50,Litri,Roma
101,2026-12-18,Rifiuti,Rifiuti speciali,427.56,kg,Roma
102,2026-12-21,Flotta aziendale,Benzina,113.91,Litri,Roma


## Phase 3 — Emissions Calculation

With a single clean dataset, we can now apply the emission factors. Each resource (Electricity, Natural Gas, Diesel…) is associated with a factor expressed in kgCO₂e per unit, sourced from ISPRA 2023, IPCC AR6 and AIB 2023. Multiplying quantity by the corresponding factor produces the `kgCO₂e` column — the core output of the GHG inventory.